In [1]:
# Chat Completion API
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

def get_chat_completion(prompt, model='gpt-5-mini'):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":"당신은 친절하고 도움이 되는 AI 비서입니다."},
            {"role":"user","content":prompt}
        ]
    )
    return response.choices[0].message.content


if __name__ == "__main__":
    user_prompt = input("AI에게 물어볼 질문을 입력하세요: ")
    reponse = get_chat_completion(user_prompt)
    print("\nAI 응답: ")
    print(reponse)


AI 응답: 
안녕 Dohy! 반가워 — 나는 AI 비서야. 무엇을 도와줄까?


In [2]:
# response API
client = OpenAI()
def get_responses(prompt, model='gpt-5-mini'):
    response = client.responses.create(
        model=model,
        tools=[{"type":"web_search_preview"}],
        input=prompt
    )
    return response.output_text

if __name__ == "__main__":
    prompt="""
    https://platform.openai.com/docs/api-reference/responses/create
    를 읽어서 Response API에 대해 3줄 요약 정리해주세요.
    """
    output = get_responses(prompt)
    print(output)

POST /responses 엔드포인트로 모델 응답을 생성하며, 텍스트·이미지 입력을 받아 텍스트 또는 구조화된 JSON 출력을 반환합니다.  
모델이 사용자 함수(함수 호출)나 웹/파일 검색 같은 내장 도구를 사용하도록 구성할 수 있고, 컨텍스트 관리, 스트리밍, 백그라운드 실행 옵션을 지원합니다.  
요청 바디에서 model, inputs, tools/functions, stream, background, store, context_management 등 다양한 파라미터로 출력 형식과 실행 방식을 세밀하게 제어할 수 있습니다. ([platform.openai.com](https://platform.openai.com/docs/api-reference/responses/create))


In [ ]:
# Anthropic
from anthropic import Anthropic

client = Anthropic()
conversation = [] # 대화 기록 저장 리스트
conversation.append({"role":"user","content":"안녕 나는 Dohy야"})

response = client.messages.create(
    model='claude-haiku-4-5',
    max_tokens=1000,
    messages=conversation
)

assistant_message = response.content[0].text #  답변 후보 목록이 아니라, 답변 하나를 이루는 콘텐츠 블록들의 목록
print(assistant_message)
conversation.append({"role":"assistant", 'content':assistant_message})
conversation.append({"role":"user", "content":"내 이름이 뭐라고?"})

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1000,
    messages=conversation
)

print(response.content[0].text)

안녕하세요, Dohy님! 만나서 반갑습니다 👋

저는 Claude라고 합니다. 무엇을 도와드릴까요?
당신의 이름은 **Dohy**라고 하셨습니다! 😊


In [ ]:
# OpenAI Stream
from openai import OpenAI
import rich

client = OpenAI()
default_model = 'gpt-5-mini'

def stream_chat_completion(prompt, model):
    stream = client.chat.completions.create(
        model=model,
        messages=[{"role":"user", "content":prompt}],
        stream=True
    )
    for chunk in stream:
        content = chunk.choices[0].delta.content # delta: 이번 스트림에서 생성된 문자
        if content is not None:
            print(content, end="")

def stream_response(prompt, model):
    with client.responses.stream(model=model, input=prompt) as stream: # 컨택스트 매니저
        for event in stream: # event: 타입이 붙은 의미 단위 이벤트(응답이 만들어지는 도중에 일어난 사건 하나하나를 알려주는 알림 객체)
            if "output_text" in event.type:
                rich.print(event)
    rich.print(stream.get_final_response())

if __name__ == "__main__":
    stream_chat_completion("스트리밍이 뭔가요?", default_model)
    stream_response("점심 메뉴를 추천해주세요.", default_model)

간단히 말하면 스트리밍은 데이터를 전부 내려받지 않고도 “흐르는 대로” 바로 재생하거나 처리하는 방식입니다.

핵심 아이디어
- 재생(또는 처리)을 위해 파일 전체를 먼저 받지 않고, 작은 조각(패킷, 세그먼트)을 순차적으로 받아 바로 사용합니다.
- 네트워크를 통해 실시간으로 전송되므로 재생 시작이 빠르고 저장 공간을 적게 씁니다.

주요 사용 사례
- 미디어 스트리밍: 유튜브·넷플릭스·스포티파이처럼 동영상·음악을 인터넷으로 실시간 재생. (온디맨드 스트리밍 vs 라이브 스트리밍)
- 데이터 스트리밍(실시간 처리): 센서·로그·금융거래 등 지속적으로 들어오는 데이터를 실시간으로 분석·처리(Kafka, Flink, Spark Streaming 등).

어떻게 동작하나
- 서버가 데이터 조각(세그먼트)을 연속으로 보냄 → 클라이언트는 받은 조각을 버퍼에 쌓아 바로 재생.
- 네트워크 상황에 따라 화질을 자동으로 바꾸는 적응형 비트레이트(예: HLS, DASH)를 사용.
- 라이브 스트리밍은 지연(latency) 관리가 중요(프로토콜: RTMP, WebRTC 등).

장단점
- 장점: 즉시 재생 가능, 기기 저장 공간 절약, 콘텐츠 업데이트 쉬움(라이브).
- 단점: 인터넷 품질에 민감(버퍼링·화질 저하), 라이브의 경우 지연 발생 가능, 오프라인 재생 어려움(오프라인 저장 기능 제외).

원하시면 더 자세히 설명해 드릴게요:
- 미디어 스트리밍의 기술(코덱, HLS/DASH, 버퍼링)  
- 라이브 스트리밍 구축 방법(장비·프로토콜)  
- 프로그래밍에서의 스트리밍(스트림 API, Kafka 등) 중 어떤 걸 원하시나요?

ResponseTextDeltaEvent(
    content_index=0,
    delta='좋',
    item_id='msg_01a9a20b06e18a74006a4011b427b88198aecd64783d5efc58',
    logprobs=[],
    output_index=1,
    sequence_number=6,
    type='response.output_text.delta',
    snapshot='좋'
)

ResponseTextDeltaEvent(
    content_index=0,
    delta='습니다',
    item_id='msg_01a9a20b06e18a74006a4011b427b88198aecd64783d5efc58',
    logprobs=[],
    output_index=1,
    sequence_number=7,
    type='response.output_text.delta',
    snapshot='좋습니다'
)

ResponseTextDeltaEvent(
    content_index=0,
    delta=' —',
    item_id='msg_01a9a20b06e18a74006a4011b427b88198aecd64783d5efc58',
    logprobs=[],
    output_index=1,
    sequence_number=8,
    type='response.output_text.delta',
    snapshot='좋습니다 —'
)

ResponseTextDeltaEvent(
    content_index=0,
    delta=' 몇',
    item_id='msg_01a9a20b06e18a74006a4011b427b88198aecd64783d5efc58',
    logprobs=[],
    output_index=1,
    sequence_number=9,
    type='response.output_text.delta',
    snapshot='좋습니다 — 몇'
)

...

ResponseTextDeltaEvent(
    content_index=0,
    delta='릴',
    item_id='msg_01a9a20b06e18a74006a4011b427b88198aecd64783d5efc58',
    logprobs=[],
    output_index=1,
    sequence_number=388,
    type='response.output_text.delta',
    snapshot='좋습니다 — 몇 가지 상황별 점심 메뉴를 추천드릴게요. 원하시는 스타일(간단·빠름 / 든든·한식 / 
가벼움·건강 / 채식 등)을 알려주시면 더 맞춤으로 골라드릴게요.\n\n1) 빠르고 간단 (15–20분)\n- 김밥 + 라볶이(또는 
컵라면): 배부르고 이동 중에도 OK.\n- 샌드위치(치킨/에그/아보카도) + 과일: 가볍고 준비 쉬움.\n\n2) 든든한 
한식(20–40분)\n- 제육볶음 덮밥: 매콤하게 밥 한 공기 뚝딱.\n- 김치찌개 + 공기밥: 추운 날이나 속 든든히 채우고 싶을 
때.\n\n3) 가벼운/건강식(10–20분)\n- 닭가슴살 샐러드 + 현미밥: 칼로리 조절 중일 때.\n- 포케볼(연어/참치): 신선하고 
영양 균형 좋음.\n\n4) 외국식/특별한 기분(20–40분)\n- 파스타(크림/토마토): 간단하지만 만족감 높음.\n- 
카레라이스(야채/치킨): 만들기 쉬우면서 든든.\n\n5) 채식 또는 비건 옵션\n- 비건 버거 또는 두부 스테이크 + 샐러드\n- 
야채 비빔밥(계란 제외 시 비건): 채소맛 풍부.\n\n원하시면\n- 오늘의 날씨·기분·식사예산·알레르기/기피 음식 알려주시면
더 구체적으로 추천해드릴게요. 어느 쪽으로 도와드릴'
)

ResponseTextDeltaEvent(
    content_index=0,
    delta='까요',
    item_id='msg_01a9a20b06e18a74006a4011b427b88198aecd64783d5efc58',
    logprobs=[],
    output_index=1,
    sequence_number=389,
    type='response.output_text.delta',
    snapshot='좋습니다 — 몇 가지 상황별 점심 메뉴를 추천드릴게요. 원하시는 스타일(간단·빠름 / 든든·한식 / 
가벼움·건강 / 채식 등)을 알려주시면 더 맞춤으로 골라드릴게요.\n\n1) 빠르고 간단 (15–20분)\n- 김밥 + 라볶이(또는 
컵라면): 배부르고 이동 중에도 OK.\n- 샌드위치(치킨/에그/아보카도) + 과일: 가볍고 준비 쉬움.\n\n2) 든든한 
한식(20–40분)\n- 제육볶음 덮밥: 매콤하게 밥 한 공기 뚝딱.\n- 김치찌개 + 공기밥: 추운 날이나 속 든든히 채우고 싶을 
때.\n\n3) 가벼운/건강식(10–20분)\n- 닭가슴살 샐러드 + 현미밥: 칼로리 조절 중일 때.\n- 포케볼(연어/참치): 신선하고 
영양 균형 좋음.\n\n4) 외국식/특별한 기분(20–40분)\n- 파스타(크림/토마토): 간단하지만 만족감 높음.\n- 
카레라이스(야채/치킨): 만들기 쉬우면서 든든.\n\n5) 채식 또는 비건 옵션\n- 비건 버거 또는 두부 스테이크 + 샐러드\n- 
야채 비빔밥(계란 제외 시 비건): 채소맛 풍부.\n\n원하시면\n- 오늘의 날씨·기분·식사예산·알레르기/기피 음식 알려주시면
더 구체적으로 추천해드릴게요. 어느 쪽으로 도와드릴까요'
)

ResponseTextDeltaEvent(
    content_index=0,
    delta='?',
    item_id='msg_01a9a20b06e18a74006a4011b427b88198aecd64783d5efc58',
    logprobs=[],
    output_index=1,
    sequence_number=390,
    type='response.output_text.delta',
    snapshot='좋습니다 — 몇 가지 상황별 점심 메뉴를 추천드릴게요. 원하시는 스타일(간단·빠름 / 든든·한식 / 
가벼움·건강 / 채식 등)을 알려주시면 더 맞춤으로 골라드릴게요.\n\n1) 빠르고 간단 (15–20분)\n- 김밥 + 라볶이(또는 
컵라면): 배부르고 이동 중에도 OK.\n- 샌드위치(치킨/에그/아보카도) + 과일: 가볍고 준비 쉬움.\n\n2) 든든한 
한식(20–40분)\n- 제육볶음 덮밥: 매콤하게 밥 한 공기 뚝딱.\n- 김치찌개 + 공기밥: 추운 날이나 속 든든히 채우고 싶을 
때.\n\n3) 가벼운/건강식(10–20분)\n- 닭가슴살 샐러드 + 현미밥: 칼로리 조절 중일 때.\n- 포케볼(연어/참치): 신선하고 
영양 균형 좋음.\n\n4) 외국식/특별한 기분(20–40분)\n- 파스타(크림/토마토): 간단하지만 만족감 높음.\n- 
카레라이스(야채/치킨): 만들기 쉬우면서 든든.\n\n5) 채식 또는 비건 옵션\n- 비건 버거 또는 두부 스테이크 + 샐러드\n- 
야채 비빔밥(계란 제외 시 비건): 채소맛 풍부.\n\n원하시면\n- 오늘의 날씨·기분·식사예산·알레르기/기피 음식 알려주시면
더 구체적으로 추천해드릴게요. 어느 쪽으로 도와드릴까요?'
)

ResponseTextDoneEvent[TypeVar](
    content_index=0,
    item_id='msg_01a9a20b06e18a74006a4011b427b88198aecd64783d5efc58',
    logprobs=[],
    output_index=1,
    sequence_number=391,
    text='좋습니다 — 몇 가지 상황별 점심 메뉴를 추천드릴게요. 원하시는 스타일(간단·빠름 / 든든·한식 / 가벼움·건강 /
채식 등)을 알려주시면 더 맞춤으로 골라드릴게요.\n\n1) 빠르고 간단 (15–20분)\n- 김밥 + 라볶이(또는 컵라면): 배부르고
이동 중에도 OK.\n- 샌드위치(치킨/에그/아보카도) + 과일: 가볍고 준비 쉬움.\n\n2) 든든한 한식(20–40분)\n- 제육볶음 
덮밥: 매콤하게 밥 한 공기 뚝딱.\n- 김치찌개 + 공기밥: 추운 날이나 속 든든히 채우고 싶을 때.\n\n3) 
가벼운/건강식(10–20분)\n- 닭가슴살 샐러드 + 현미밥: 칼로리 조절 중일 때.\n- 포케볼(연어/참치): 신선하고 영양 균형 
좋음.\n\n4) 외국식/특별한 기분(20–40분)\n- 파스타(크림/토마토): 간단하지만 만족감 높음.\n- 카레라이스(야채/치킨): 
만들기 쉬우면서 든든.\n\n5) 채식 또는 비건 옵션\n- 비건 버거 또는 두부 스테이크 + 샐러드\n- 야채 비빔밥(계란 제외 
시 비건): 채소맛 풍부.\n\n원하시면\n- 오늘의 날씨·기분·식사예산·알레르기/기피 음식 알려주시면 더 구체적으로 
추천해드릴게요. 어느 쪽으로 도와드릴까요?',
    type='response.output_text.done',
    parsed=None
)

ParsedResponse[TypeVar](
    id='resp_01a9a20b06e18a74006a4011b0de188198bb0927098d784c50',
    created_at=1782583728.0,
    error=None,
    incomplete_details=None,
    instructions=None,
    metadata={},
    model='gpt-5-mini-2025-08-07',
    object='response',
    output=[
        ResponseReasoningItem(
            id='rs_01a9a20b06e18a74006a4011b1356081989031ed16e487112a',
            summary=[],
            type='reasoning',
            content=[],
            encrypted_content=None,
            status=None
        ),
        ParsedResponseOutputMessage[TypeVar](
            id='msg_01a9a20b06e18a74006a4011b427b88198aecd64783d5efc58',
            content=[
                ParsedResponseOutputText[TypeVar](
                    annotations=[],
                    text='좋습니다 — 몇 가지 상황별 점심 메뉴를 추천드릴게요. 원하시는 스타일(간단·빠름 / 든든·한식
/ 가벼움·건강 / 채식 등)을 알려주시면 더 맞춤으로 골라드릴게요.\n\n1) 빠르고 간단 (15–20분)\n- 김밥 + 라볶이(또는 
컵라면): 배부르고 이동 중에도 OK.\n- 샌드위치(치킨/에그/아보카도) + 과일: 가볍고 준비 쉬움.\n\n2) 든든한 
한식(20–40분)\n- 제육볶음 덮밥: 매콤하게 밥 한 공기 뚝딱.\n- 김치찌개 + 공기밥: 추운 날이나 속 든든히 채우고 싶을 
때.\n\n3) 가벼운/건강식(10–20분)\n- 닭가슴살 샐러드 + 현미밥: 칼로리 조절 중일 때.\n- 포케볼(연어/참치): 신선하고 
영양 균형 좋음.\n\n4) 외국식/특별한 기분(20–40분)\n- 파스타(크림/토마토): 간단하지만 만족감 높음.\n- 
카레라이스(야채/치킨): 만들기 쉬우면서 든든.\n\n5) 채식 또는 비건 옵션\n- 비건 버거 또는 두부 스테이크 + 샐러드\n- 
야채 비빔밥(계란 제외 시 비건): 채소맛 풍부.\n\n원하시면\n- 오늘의 날씨·기분·식사예산·알레르기/기피 음식 알려주시면
더 구체적으로 추천해드릴게요. 어느 쪽으로 도와드릴까요?',
                    type='output_text',
                    logprobs=[],
                    parsed=None
                )
            ],
            role='assistant',
            status='completed',
            type='message',
            phase=None
        )
    ],
    parallel_tool_calls=True,
    temperature=1.0,
    tool_choice='auto',
    tools=[],
    top_p=1.0,
    background=False,
    completed_at=1782583736.0,
    conversation=None,
    max_output_tokens=None,
    max_tool_calls=None,
    previous_response_id=None,
    prompt=None,
    prompt_cache_key=None,
    prompt_cache_retention='in_memory',
    reasoning=Reasoning(
        effort='medium',
        generate_summary=None,
        summary=None,
        context='current_turn',
        mode='standard'
    ),
    safety_identifier=None,
    service_tier='default',
    status='completed',
    text=ResponseTextConfig(format=ResponseFormatText(type='text'), verbosity='medium'),
    top_logprobs=0,
    truncation='disabled',
    usage=ResponseUsage(
        input_tokens=13,
        input_tokens_details=InputTokensDetails(cached_tokens=0),
        output_tokens=717,
        output_tokens_details=OutputTokensDetails(reasoning_tokens=256),
        total_tokens=730
    ),
    user=None,
    frequency_penalty=0.0,
    moderation=None,
    presence_penalty=0.0,
    store=True,
    tool_usage={
        'image_gen': {
            'input_tokens': 0,
            'input_tokens_details': {'image_tokens': 0, 'text_tokens': 0},
            'output_tokens': 0,
            'output_tokens_details': {'image_tokens': 0, 'text_tokens': 0},
            'total_tokens': 0
        },
        'web_search': {'num_requests': 0}
    }
)

In [6]:
# Anthropic Stream
from anthropic import Anthropic
import rich

client = Anthropic()
prompt="Anthropic의 발음은 앤트로픽이 맞나요, 앤쓰로픽이 맞나요?"
with client.messages.stream(
    max_tokens=1024,
    messages=[{"role":"user","content":prompt}],
    model='claude-haiku-4-5-20251001',
) as stream:
    for event in stream:
        if event.type == "text":
            print(event.text, end="")
    print()

    rich.print(stream.get_final_message())

# Anthropic 발음

**앤트로픽**이 맞습니다.

영어 발음 기준으로 /ænˈθrɒpɪk/ 인데, 한글로 옮길 때:
- **th**는 보통 '드' 또는 '쓰' 음으로 표기하지만
- 이 경우 **'트'**로 표기하는 것이 자연스럽습니다

따라서 **"앤트로픽"** 또는 **"앤쓰로픽"** 모두 사용되긴 하지만, 더 일반적이고 자연스러운 표기는 **앤트로픽**입니다.


ParsedMessage(
    id='msg_011Cd3ZtPszskrAu4Mkv1NF6',
    container=None,
    content=[
        ParsedTextBlock(
            citations=None,
            text='# Anthropic 발음\n\n**앤트로픽**이 맞습니다.\n\n영어 발음 기준으로 /ænˈθrɒpɪk/ 인데, 한글로 옮길 
때:\n- **th**는 보통 \'드\' 또는 \'쓰\' 음으로 표기하지만\n- 이 경우 **\'트\'**로 표기하는 것이 
자연스럽습니다\n\n따라서 **"앤트로픽"** 또는 **"앤쓰로픽"** 모두 사용되긴 하지만, 더 일반적이고 자연스러운 표기는 
**앤트로픽**입니다.',
            type='text',
            parsed_output=None
        )
    ],
    model='claude-haiku-4-5-20251001',
    role='assistant',
    stop_details=None,
    stop_reason='end_turn',
    stop_sequence=None,
    type='message',
    usage=Usage(
        cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
        cache_creation_input_tokens=0,
        cache_read_input_tokens=0,
        inference_geo='not_available',
        input_tokens=45,
        output_tokens=207,
        output_tokens_details=None,
        server_tool_use=None,
        service_tier='standard'
    )
)

In [2]:
# Anthropioc Stream
import anthropic
import rich

client = anthropic.Anthropic()
prompt = "anthropic 발음은 앤트로픽이 맞나요? 앤트로픽이 맞나요?"
with client.messages.stream(
    max_tokens = 1024,
    messages=[{'role':'user','content':prompt}],
    model='claude-haiku-4-5'
) as stream:
    for event in stream:
        if event.type == "text":
            print(event.text, end='')
    print()
    rich.print(stream.get_final_message())

# Anthropic 발음

**앤트로픽**이 맞습니다.

## 발음 분석
- **영어 원문**: Anthropic /ænˈθrɑːpɪk/
- **한글 표기**: 앤트로픽

각 음절을 풀어서 설명하면:
- **An**-: 앤
- **throp**-: 트로
- **-ic**: 픽

한국에서는 보통 "앤트로픽" 또는 "앤써로픽"으로 표기하기도 하지만, **앤트로픽**이 가장 표준적인 표기입니다.


ParsedMessage(
    id='msg_011Cd5jjx7eXbQEmqEdboK8E',
    container=None,
    content=[
        ParsedTextBlock(
            citations=None,
            text='# Anthropic 발음\n\n**앤트로픽**이 맞습니다.\n\n## 발음 분석\n- **영어 원문**: Anthropic 
/ænˈθrɑːpɪk/\n- **한글 표기**: 앤트로픽\n\n각 음절을 풀어서 설명하면:\n- **An**-: 앤\n- **throp**-: 트로\n- 
**-ic**: 픽\n\n한국에서는 보통 "앤트로픽" 또는 "앤써로픽"으로 표기하기도 하지만, **앤트로픽**이 가장 표준적인 
표기입니다.',
            type='text',
            parsed_output=None
        )
    ],
    model='claude-haiku-4-5-20251001',
    role='assistant',
    stop_details=None,
    stop_reason='end_turn',
    stop_sequence=None,
    type='message',
    usage=Usage(
        cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
        cache_creation_input_tokens=0,
        cache_read_input_tokens=0,
        inference_geo='not_available',
        input_tokens=41,
        output_tokens=192,
        output_tokens_details=None,
        server_tool_use=None,
        service_tier='standard'
    )
)

In [ ]:
# Asynchronous 
import asyncio
from openai import AsyncOpenAI
from anthropic import AsyncAnthropic

openai_client = AsyncOpenAI()
claude_client = AsyncAnthropic()

async def call_async_openai(prompt: str, model: str="gpt-5-mini")->str:
    response = await openai_client.chat.completions.create( # await를 사용해 비동기적으로 API 응답을 기다림
        model=model,
        messages=[{'role':"user","content":prompt}],
    )
    return response.choices[0].message.content

async def call_async_claude(prompt: str, model: str= "claude-haiku-4-5-20251001")->str:
    response = await claude_client.messages.create( # await를 사용해 비동기적으로 API 응답을 기다림
        model=model, max_tokens=1000, messages=[{'role':'user','content':prompt}]
    )
    return response.content[0].text

async def main():
    print("동시에 API 호츨")
    prompt = "비동기 프로그래밍에 대해 두세 문장으로 설명해주세요."
    # 비동기 함수 호출 시, 코루틴(멈췄다 이어갈 수 있는 함수) 객체 반환(실행은 아직X)
    openai_task = call_async_openai(prompt)
    claude_task = call_async_claude(prompt)

    # 두 API 호출을 동시에 실행하고, 둘 다 완료될 때까지 대기
    openai_response, cluade_response = await asyncio.gather(openai_task, claude_task) # gather: 여러 코루틴을 동시에 실행하고 모든 결과를 기다림
    print(f"OpenAI 응답: {openai_response}")
    print(f"Claude 응답: {cluade_response}")

if __name__ == "__main__":
    #asyncio.run(main()) # 비동기 메인 함수를 이벤트 루프에서 실행
    await main() 



동시에 API 호츨
OpenAI 응답: 비동기 프로그래밍은 작업을 요청한 뒤 결과를 기다리지 않고 다른 일을 계속 수행하게 해 지연이 큰 입출력·네트워크 작업을 효율적으로 처리하는 방식입니다. 콜백·프라미스(또는 미래)·async/await 같은 기법으로 결과를 다루며, 응답성과 자원 활용을 높이는 대신 동시성 관리와 에러 처리의 복잡성이 커질 수 있습니다.
Claude 응답: # 비동기 프로그래밍 설명

**비동기 프로그래밍**은 어떤 작업이 완료될 때까지 기다리지 않고 다른 작업을 계속 진행하는 방식입니다. 예를 들어 파일 읽기나 네트워크 요청 같이 시간이 걸리는 작업을 백그라운드에서 처리하면서, 프로그램은 다른 코드를 실행할 수 있습니다. 작업이 완료되면 콜백, Promise, async/await 등의 방식으로 결과를 처리합니다.


In [ ]:
# 간헐적으로 실패하는 함수 및 tenacity를 적용하여 재처리
import asyncio
import os
import logging
import random
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import AsyncOpenAI
from anthropic import AsyncAnthropic

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

openai_client = AsyncOpenAI()
claude_client = AsyncAnthropic()

# 테스트용 간헐적 실패 시뮬레이션 함수
async def simulate_random_failure():
    if random.random() < 0.5:
        logger.warning("인위적으로 API 호출 실패 발생")
        raise ConnectionError("인위적으로 발생시킨 연결 오류") # 예외를 반환하여 @retry가 동작하도록 함
    # 약간의 지연시간 추가
    await asyncio.sleep(random.uniform(0.1, 0.5)) # 오류가 안났을 시, 0.1~0.5초 중 임의로 지연


# tenacity를 사용한 재시도 데코레이터 적용
@retry(stop=stop_after_attempt(3), # 최대 3번시도
       wait=wait_exponential(multiplier=1,min=2,max=10), # 지수 백오프: 2초(min(2^0,2)), 4초, 8초 ...
       retry=retry_if_exception_type(), # 모든 예외에 대해 재시도
       before_sleep = lambda retry_state: logger.warning(
           f"API 호출 실패:{retry_state.outcome.exception()}, {retry_state.attempt_number}번째 재시도중.."
       ) # before_sleep: 재시도 전 대기 직전에 실행할 함수
    )

async def call_async_openai(prompt: str, model: str = 'gpt-5-mini')->str: # 랜덤한 확률로 실패하는 함수
    logger.info(f"OpenAI API 호출 시작: {model}")

    await simulate_random_failure() # 테스트를 위한 랜덤 실패 시뮬레이션
    response = await openai_client.chat.completions.create(
        model=model,
        messages=[{'role':"user","content":prompt}],
    )
    logger.info("OpenAI API 호출 성공")
    return response.choices[0].message.content

async def call_async_claude(prompt: str, model: str = 'claude-haiku-4-5') ->str:
    logger.info(f"Claude API 호출 시작: {model}")
    response = await claude_client.messages.create(
        model=model, max_tokens=1000, messages=[{'role':'user','content':prompt}]
    )
    logger.info("Claude API 호출 성공")
    return response.content[0].text

async def main():
    print("동시에 API 호출하기 (재시도 로직 포함)")
    prompt = "비동기 프로그래밍에 대해서 2~3줄로 설명해줘"
    openai_task = call_async_openai(prompt)
    claude_task = call_async_claude(prompt)

    try:
        openai_response, claude_response = await asyncio.gather( # return_exception = False : 하나가 실패하면 최종 실패, True면 하나가 실패해도 예외가 결과 리스트에 담겨 반환
            openai_task, claude_task, return_exceptions=False
        )
        print(f"OpenAI 응답: {openai_response}")
        print(f"Claude 응답: {claude_response}")

    except Exception as e:
        logger.error(f"API 호출 중 처리되지 않은 오류 발생: {e}")

if __name__=="__main__":
    await main()

INFO:__main__:OpenAI API 호출 시작: gpt-5-mini
INFO:__main__:Claude API 호출 시작: claude-haiku-4-5


동시에 API 호출하기 (재시도 로직 포함)


INFO:__main__:OpenAI API 호출 시작: gpt-5-mini
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:__main__:Claude API 호출 성공
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:__main__:OpenAI API 호출 성공


OpenAI 응답: 비동기 프로그래밍은 작업을 시작한 뒤 완료를 기다리는 동안 다른 코드가 실행되도록 하는 방식입니다.
주로 I/O나 네트워크 호출처럼 시간이 걸리는 작업에서 프로그램의 응답성·처리량을 높여줍니다.
콜백, 프라미스(또는 미래), async/await 같은 메커니즘으로 결과를 처리합니다.
Claude 응답: # 비동기 프로그래밍

**긴 작업이 끝날 때까지 기다리지 않고**, 다른 코드를 먼저 실행하는 방식입니다. 작업이 완료되면 그 결과를 받아 처리합니다.

예를 들어 서버에서 데이터를 받아오는 동안 UI를 업데이트하거나 사용자 입력을 받을 수 있어서 **반응성 높은 프로그램**을 만들 수 있습니다.
